Task 1:Data Understanding and Exploration

In [ ]:
import pandas as pd
import numpy as np
raw_data = pd.read_csv('G2.csv')
data = raw_data.copy()
data.head()

In [ ]:
data.shape

In [ ]:
data.info()

In [ ]:
data.isna().sum()

In [ ]:
data.describe()

In [ ]:
data.drop(columns=['id'],inplace=True)
data

In [ ]:
no_duplicates = data.duplicated().sum()
no_duplicates

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

### Task 2: Data Preprocessing

This section handles missing values, encodes categorical input features, and creates the training and testing datasets before EDA. Missing `type_2` means no secondary type and is labeled `none`; numeric missing values in the cleaned EDA dataframe are filled with their medians. A zero weight is treated as invalid and missing.

For machine learning, the raw data is split first. Numeric imputation, scaling, and one-hot encoding are fitted on the training data only, then applied to the test data. This prevents information from the test set leaking into preprocessing.

In [ ]:
# Treat the single zero weight as an invalid/missing measurement.
data.loc[data['weight_hg'].eq(0), 'weight_hg'] = np.nan
missing_before = data.isna().sum()
numeric_columns = data.select_dtypes(include=np.number).columns
data[numeric_columns] = data[numeric_columns].fillna(data[numeric_columns].median())
data['type_2'] = data['type_2'].fillna('none')
print('Missing values before preprocessing:\n', missing_before[missing_before > 0])
print('Missing values after preprocessing:', int(data.isna().sum().sum()))
data.isna().sum()

#### Prepare features and create training/testing datasets

The target is `type_1`. `name` and `id` are excluded from the input features. The feature transformer imputes missing numeric values with the training median, scales numeric columns, and one-hot encodes categorical columns. The target labels remain class names for classification.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, RobustScaler

target = 'type_1'
X = raw_data.drop(columns=[target, 'name', 'id']).copy()
y = raw_data[target].copy()
X.loc[X['weight_hg'].eq(0), 'weight_hg'] = np.nan

categorical_features = X.select_dtypes(include=['object', 'category']).columns.tolist()
numeric_features = X.select_dtypes(include=np.number).columns.tolist()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

numeric_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', RobustScaler())
])
categorical_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='none')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(transformers=[
    ('numeric', numeric_pipeline, numeric_features),
    ('categorical', categorical_pipeline, categorical_features)
])

# Fit preprocessing only on training data; apply the learned transform to test data.
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print('Training feature matrix:', X_train_processed.shape)
print('Testing feature matrix: ', X_test_processed.shape)
print('Training labels:        ', y_train.shape)
print('Testing labels:         ', y_test.shape)
print('Training class counts:\n', y_train.value_counts().sort_index())
print('Testing class counts:\n', y_test.value_counts().sort_index())

The cleaned `data` dataframe is complete for EDA. `X_train_processed` and `X_test_processed` are ready-to-use encoded feature matrices; `y_train` and `y_test` contain the corresponding `type_1` labels. Model pipelines below also fit this same preprocessing on each training fold when doing cross-validation.

### Task 3: Exploratory Data Analysis

Explore the cleaned dataset and relationships between attributes. For charts that show original distributions, use `raw_data` so imputation does not distort the plot. The Base Experience comparison omits original missing observations; the Height vs Weight plot omits the one invalid zero-weight observation and uses logarithmic axes. Plausible extreme values are retained.

In [ ]:
sns.histplot(data=raw_data, x='base_experience', bins=15, kde=True)
plt.title('Base Experience Distribution (missing values omitted)')
plt.show()

In [ ]:
sns.countplot(data=data, x='type_2', order=data['type_2'].value_counts().index)
plt.title('Secondary Type Counts (including no secondary type)')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
attack_defense_rho = data[['attack', 'defense']].corr(method='spearman').iloc[0, 1]
plt.figure(figsize=(12, 7))
ax = sns.scatterplot(data=data, x='attack', y='defense', hue='type_1', palette='tab20', s=36, alpha=0.7)
ax.set_title(f'Attack vs Defense by Primary Type (Spearman correlation = {attack_defense_rho:.2f})')
ax.set_xlabel('Attack')
ax.set_ylabel('Defense')
ax.legend(title='Primary type', bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=8)
plt.tight_layout(rect=(0, 0, 0.84, 1))
plt.show()
print(f'Spearman correlation between Attack and Defense: {attack_defense_rho:.3f}')

In [ ]:
hp_type_order = data.groupby('type_1')['hp'].median().sort_values().index
plt.figure(figsize=(14, 6))
ax = sns.boxplot(data=data, x='type_1', y='hp', order=hp_type_order, color='skyblue')
ax.set_title('HP Distribution Across Primary Types')
ax.set_xlabel('Primary type (ordered by median HP)')
ax.set_ylabel('HP')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
experience_data = raw_data.dropna(subset=['base_experience'])
experience_type_order = experience_data.groupby('type_1')['base_experience'].median().sort_values().index
plt.figure(figsize=(14, 6))
ax = sns.boxplot(data=experience_data, x='type_1', y='base_experience', order=experience_type_order, color='lightgreen')
ax.set_title('Base Experience by Primary Type (missing values omitted)')
ax.set_xlabel('Primary type (ordered by median Base Experience)')
ax.set_ylabel('Base Experience')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
height_weight_data = raw_data.loc[raw_data['weight_hg'] > 0]
height_weight_rho = height_weight_data[['height_dm', 'weight_hg']].corr(method='spearman').iloc[0, 1]
fig, ax = plt.subplots(figsize=(9, 7))
sns.scatterplot(data=height_weight_data, x='height_dm', y='weight_hg', alpha=0.55, s=35, ax=ax)
ax.set_xscale('log')
ax.set_yscale('log')
ax.set_title(f'Height vs Weight (log scales; Spearman correlation = {height_weight_rho:.2f})')
ax.set_xlabel('Height (dm, log scale)')
ax.set_ylabel('Weight (hg, log scale)')
ax.grid(True, which='both', alpha=0.2)
plt.tight_layout()
plt.show()
print(f'Spearman correlation between Height and Weight (positive weights): {height_weight_rho:.3f}')

In [ ]:
plt.boxplot(data['base_experience'])
plt.xlabel('base_experience	')
plt.title('Box Plot of base_experience	')
plt.show()

In [ ]:
plt.boxplot(data['height_dm'])
plt.xlabel('height_dm	')
plt.title('Box Plot of height_dm')
plt.show()

In [ ]:
plt.boxplot(data['weight_hg'])
plt.xlabel('weight_hg	')
plt.title('Box Plot of weight_hg')
plt.show()

### Task 4: Machine Learning Modeling

Train four classifiers to predict `type_1`: Logistic Regression, Random Forest, RBF Support Vector Classifier, and K-Nearest Neighbors. Compare accuracy with stratified cross-validation on the training data, then evaluate on the held-out test data. Select the model with the highest cross-validation accuracy; this avoids choosing a model based on test results. Macro F1 and weighted F1 are included to show performance across imbalanced type classes.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC

models = {
    'Logistic Regression': LogisticRegression(max_iter=3000, class_weight='balanced'),
    # No class weighting is used because the goal here is to improve overall accuracy.
    'Random Forest': RandomForestClassifier(
        n_estimators=400, max_features='sqrt', class_weight=None,
        random_state=42, n_jobs=-1
    ),
    'RBF SVC': SVC(kernel='rbf', class_weight='balanced'),
    'K-Nearest Neighbors': KNeighborsClassifier(n_neighbors=7, weights='distance')
}

model_pipelines = {
    name: Pipeline(steps=[('preprocessor', preprocessor), ('classifier', estimator)])
    for name, estimator in models.items()
}

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
evaluation_rows = []
for name, pipeline in model_pipelines.items():
    cv_accuracy = cross_val_score(pipeline, X_train, y_train, cv=cv, scoring='accuracy').mean()
    pipeline.fit(X_train, y_train)
    predictions = pipeline.predict(X_test)
    evaluation_rows.append({
        'Model': name,
        'CV Accuracy': cv_accuracy,
        'Test Accuracy': accuracy_score(y_test, predictions),
        'Precision (macro)': precision_score(y_test, predictions, average='macro', zero_division=0),
        'Recall (macro)': recall_score(y_test, predictions, average='macro', zero_division=0),
        'F1 (macro)': f1_score(y_test, predictions, average='macro', zero_division=0),
        'F1 (weighted)': f1_score(y_test, predictions, average='weighted', zero_division=0)
    })

results = pd.DataFrame(evaluation_rows).sort_values(
    ['CV Accuracy', 'Test Accuracy'], ascending=False
).reset_index(drop=True)
display(results.round(3))
best_model_name = results.loc[0, 'Model']
best_model = model_pipelines[best_model_name]
best_predictions = best_model.predict(X_test)
print(f"Selected model by cross-validation accuracy: {best_model_name}")
print(f"Held-out test accuracy: {results.loc[0, 'Test Accuracy']:.3f}")

#### Best model: confusion matrix

The selected model is chosen by cross-validation accuracy on the training split. Its confusion matrix below is calculated on the separate held-out test set. Rows are true types and columns are predicted types.


In [ ]:
from sklearn.metrics import confusion_matrix

class_labels = sorted(y.unique())
matrix = confusion_matrix(y_test, best_predictions, labels=class_labels)
fig, ax = plt.subplots(figsize=(13, 11))
sns.heatmap(matrix, annot=True, fmt='d', cmap='Blues', xticklabels=class_labels, yticklabels=class_labels, ax=ax)
ax.set_title(f'Confusion Matrix - {best_model_name}')
ax.set_xlabel('Predicted type_1')
ax.set_ylabel('True type_1')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

#### Most influential feature

Permutation importance is calculated on the held-out test set using accuracy. Importance is shown for the original input columns (rather than individual one-hot encoded categories). A larger accuracy decrease after shuffling a feature indicates that the selected model relies more on it; this does not establish causation.


In [ ]:
from sklearn.inspection import permutation_importance

permutation_result = permutation_importance(
    best_model, X_test, y_test, scoring='accuracy', n_repeats=10, random_state=42, n_jobs=-1
)
feature_importance = pd.DataFrame({
    'Feature': X_test.columns,
    'Mean accuracy decrease': permutation_result.importances_mean,
    'Standard deviation': permutation_result.importances_std
}).sort_values('Mean accuracy decrease', ascending=False).reset_index(drop=True)
display(feature_importance.round(4))
top_feature = feature_importance.iloc[0]['Feature']
print(f"Most influential feature for {best_model_name}: {top_feature} (mean accuracy decrease = {feature_importance.iloc[0]['Mean accuracy decrease']:.4f})")

plot_data = feature_importance.sort_values('Mean accuracy decrease')
plt.figure(figsize=(9, 5))
plt.barh(plot_data['Feature'], plot_data['Mean accuracy decrease'], xerr=plot_data['Standard deviation'], color='teal', alpha=0.8)
plt.xlabel('Decrease in accuracy after permutation')
plt.title(f'Permutation Feature Importance - {best_model_name}')
plt.tight_layout()
plt.show()